# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer - Exploration with `mlcroissant`

This notebook guides users in loading, exploring, and processing the FAIR² dataset on clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors using the `mlcroissant` library and Croissant metadata.

### Dataset Source
The dataset source is defined via the Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install --quiet mlcroissant

## 1. Data Loading

Load Croissant metadata and records from the FAIR² dataset using `mlcroissant`. This step retrieves structured metadata and enables programmatic access to record sets for further analysis.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and display basic information
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset loaded successfully.")
print(f"\nTitle: {metadata.name}")
print(f"Description: {metadata.description}\n")

## 2. Data Overview

Review available record sets, fields, and their unique `@id` values. The `@id` field uniquely identifies record sets and their constituent fields/columns. We'll print record set names, `@id` values, and display their fields and field IDs for reference.

In [ ]:
# List all record sets and their fields using @id for precise referencing
record_sets = dataset.metadata.record_sets
if not record_sets:
    print("No record sets found in this Croissant metadata. Please check or update the dataset schema.")
else:
    print(f"Total record sets: {len(record_sets)}\n")
    for rs in record_sets:
        print(f"Record Set: {rs.name}\n  @id: {rs.id}")
        if rs.fields:
            print("  Fields and columns:")
            for f in rs.fields:
                print(f"    - Field: {getattr(f,'name',None)}\n      @id: {getattr(f, 'id', None)}")
        else:
            print("  [No fields found]")
        print("\n")
# Optionally, list the @id's for later use
recordset_ids = [rs.id for rs in record_sets]

## 3. Data Extraction

Load all data from selected record sets into pandas DataFrames for analysis. Reference each record set and its fields by their `@id`. This makes further data operations unambiguous and reproducible.

We demonstrate extraction for all available record sets.

In [ ]:
# Extract data from all record sets into separate DataFrames
dataframes = dict()

for rs_id in recordset_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded '{rs_id}': {df.shape[0]} rows, {df.shape[1]} columns.")
    except Exception as e:
        print(f"Could not load records for record set {rs_id}: {e}")

# For demonstration, display the columns and head of the first available record set
if len(dataframes) > 0:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns for record set '@id': {first_rs}")
    pprint(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()

## 4. Exploratory Data Analysis (EDA)

Process the data to gain insights:
- Filter records based on criteria (e.g., threshold on a numeric field)
- Normalize a numeric field
- Group by a categorical field and aggregate

All references to fields/columns use their `@id` to ensure clarity and reproducibility. Please update the `numeric_field_id` and `group_field_id` variables below based on the output from the previous cell if needed.

In [ ]:
# EDA: Specify record set, numeric, and group field by their @id

# Replace these values based on output from cell [2]. Example values shown:
record_set_id = recordset_ids[0] if recordset_ids else None  # Use first record set

# For this example, guess numeric and group field @ids (update as needed!)
# You may run dataframes[record_set_id].columns to inspect possible choices
if record_set_id is not None:
    columns = dataframes[record_set_id].columns.tolist()
    print(f"Available columns in {record_set_id} =", columns)

    # Try to auto-select numeric and group fields (customize as needed):
    import re
    
    # Try to choose 'age' or similar as numeric field
    numeric_field_candidates = [col for col in columns if re.search(r'age|interval|years|count|score|number|size|length', col, re.IGNORECASE)]
    numeric_field_id = numeric_field_candidates[0] if numeric_field_candidates else columns[0] if columns else None
    
    # Try to choose 'Sex'/'Gender'/'group'/'location' as group field
    group_field_candidates = [col for col in columns if re.search(r'sex|gender|group|location|site|status|type', col, re.IGNORECASE)]
    group_field_id = group_field_candidates[0] if group_field_candidates else None
    
    print(f"\nSelected numeric field: {numeric_field_id}")
    print(f"Selected group field: {group_field_id}")

    df = dataframes[record_set_id]

    # Proceed only if the fields exist and are suitable
    if numeric_field_id is not None and numeric_field_id in df.columns:
        pd.set_option('mode.chained_assignment', None)
        
        # Coerce numeric if not already numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0

        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"\nFiltered records in '{record_set_id}' with '{numeric_field_id}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric column
        mean_ = filtered_df[numeric_field_id].mean()
        std_ = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_) / std_ if std_ else filtered_df[numeric_field_id]
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouped analysis (if group field exists)
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
    else:
        print("No suitable numeric field found in data for EDA.")
else:
    print("No available record sets; cannot perform EDA.")

## 5. Visualization

Visualize the distribution of a numeric variable or examine group-wise summaries using matplotlib or seaborn. Adjust `numeric_field_id` and `group_field_id` as appropriate for your dataset and exploration focus.

In [ ]:
# Simple histogram and boxplot for the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id and numeric_field_id in dataframes[record_set_id].columns:
    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    sns.histplot(dataframes[record_set_id][numeric_field_id].dropna(), bins=15, kde=True, ax=axs[0])
    axs[0].set_title(f"Histogram of {numeric_field_id}")

    sns.boxplot(y=dataframes[record_set_id][numeric_field_id], ax=axs[1])
    axs[1].set_title(f"Boxplot of {numeric_field_id}")
    plt.tight_layout()
    plt.show()

    # If group_field_id is available, make a boxplot by group
    if group_field_id and group_field_id in dataframes[record_set_id].columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=dataframes[record_set_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

- Using `mlcroissant`, we demonstrated how to load, examine, and process a FAIR² dataset described by a Croissant schema.
- All data entities and fields were referenced strictly by their `@id`, ensuring precise and reusable code.
- Initial EDA and visualization steps help to uncover distributions and possible patterns in clinicopathological variables of cancer survivors with second primary colorectal cancer.
- For detailed analyses, further customization (such as domain knowledge-based field selection, more advanced groupings, or modeling) can be carried out using the same `mlcroissant` interface.